# 02 · Embeddings
Run every image through frozen BiomedCLIP and DINOv3 (both ViT-B/16); save features to Drive.



## 0 · Setup

In [1]:
!pip install -q open_clip_torch pydicom "transformers>=4.56"

from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json
from pathlib import Path
import numpy as np, pandas as pd, torch

PROJECT = Path('/content/drive/MyDrive/345_term_project')
DATA    = PROJECT / 'data'
FEATS   = PROJECT / 'features'
LOCAL   = Path('/content/rsna')
FEATS.mkdir(parents=True, exist_ok=True)

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'Switch to a GPU runtime (Runtime → Change runtime type).'
print(torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
Mounted at /content/drive
Tesla T4


In [2]:
ZIP = DATA / 'rsna-pneumonia-detection-challenge.zip'
if not (LOCAL / 'stage_2_train_images').exists():
    !unzip -q -o "{ZIP}" 'stage_2_train_images/*' -d "{LOCAL}"

splits = pd.read_csv(DATA / 'splits.csv')
IMAGE_IDS = splits['image_id'].tolist()
print(len(IMAGE_IDS), 'images')

26684 images


## 1 · Image loader
DICOM → RGB. Each model applies its own resize + normalization.

In [3]:
import pydicom
from PIL import Image
from torch.utils.data import Dataset, DataLoader

def load_xray(image_id):
    ds = pydicom.dcmread(LOCAL / 'stage_2_train_images' / f'{image_id}.dcm')
    img = ds.pixel_array.astype(np.float32)
    if ds.get('PhotometricInterpretation') == 'MONOCHROME1':
        img = img.max() - img
    img = (255 * (img - img.min()) / max(img.max() - img.min(), 1)).astype(np.uint8)
    return Image.fromarray(img).convert('RGB')

class XrayDataset(Dataset):
    def __init__(self, ids, transform):
        self.ids, self.transform = ids, transform
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return self.transform(load_xray(self.ids[i]))

def loader(transform, batch_size=128):
    return DataLoader(XrayDataset(IMAGE_IDS, transform), batch_size=batch_size,
                      num_workers=os.cpu_count(), pin_memory=True)

## 2 · BiomedCLIP → 512-d image embedding

In [4]:
from open_clip import create_model_from_pretrained
from tqdm.auto import tqdm

BMC_NAME = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
BMC_OUT = FEATS / 'biomedclip_image.npy'

bmc, bmc_preprocess = create_model_from_pretrained(BMC_NAME)
bmc = bmc.to(DEVICE).eval()

feats = []
with torch.no_grad():
    for x in tqdm(loader(bmc_preprocess), desc='BiomedCLIP'):
        feats.append(bmc.encode_image(x.to(DEVICE)).cpu())
np.save(BMC_OUT, torch.cat(feats).numpy())
del bmc; torch.cuda.empty_cache()

open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  784MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

BiomedCLIP:   0%|          | 0/209 [00:00<?, ?it/s]

## 3 · DINOv3 → 768-d CLS token + 768-d patch-token mean
Output tokens are `[CLS, 4 registers, 196 patches]`.

In [5]:
from transformers import AutoImageProcessor, AutoModel

DINO_NAME = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
DINO_CLS_OUT = FEATS / 'dinov3_cls.npy'
DINO_PATCH_OUT = FEATS / 'dinov3_patchmean.npy'

processor = AutoImageProcessor.from_pretrained(DINO_NAME)
dino = AutoModel.from_pretrained(DINO_NAME).to(DEVICE).eval()

def dino_preprocess(img):
    return processor(images=img, return_tensors='pt')['pixel_values'][0]

cls_feats, patch_feats = [], []
with torch.no_grad():
    for x in tqdm(loader(dino_preprocess), desc='DINOv3'):
        tokens = dino(pixel_values=x.to(DEVICE)).last_hidden_state
        cls_feats.append(tokens[:, 0].cpu())
        patch_feats.append(tokens[:, 5:].mean(1).cpu())
np.save(DINO_CLS_OUT, torch.cat(cls_feats).numpy())
np.save(DINO_PATCH_OUT, torch.cat(patch_feats).numpy())
del dino; torch.cuda.empty_cache()

preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  343MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

DINOv3:   0%|          | 0/209 [00:00<?, ?it/s]

## 4 · Checks

In [6]:
(FEATS / 'image_ids.json').write_text(json.dumps(IMAGE_IDS))

features = {
    'biomedclip':       np.load(BMC_OUT),
    'dinov3_cls':       np.load(DINO_CLS_OUT),
    'dinov3_patchmean': np.load(DINO_PATCH_OUT),
}
for name, X in features.items():
    assert len(X) == len(IMAGE_IDS), name
    assert np.isfinite(X).all(), f'{name} has NaN/inf'
    print(f'{name:18s} {X.shape}  mean |x| = {np.abs(X).mean():.3f}')

biomedclip         (26684, 512)  mean |x| = 2.608
dinov3_cls         (26684, 768)  mean |x| = 0.484
dinov3_patchmean   (26684, 768)  mean |x| = 0.294
